# TopicBank: Bank Creation Experiment

Here we are going to collect interpretable topics (automatically, using topic coherence) from multiple model training.
These topics constitute *topic bank*.
And then the topic bank is going to be used for estimating topic models quality in the notebook [TopicBank-Experiment: Model Validation](TopicBank-Experiment-ModelValidation.ipynb).

The process is repeated for several datasets (some of them are already downloadable using [TopicNet](https://github.com/machine-intelligence-laboratory/TopicNet) library).

# Contents<a id="contents"></a>

* [Data](#data)
    * [Coocs](#coocs)
        * [Lower Memory Consumption (or a Bit of Shamanism. Part 1)](#optimizing-memory)
    * [Documents for Coherence Scores](#docs-for-cohs)
        * [Lower Time Consumption in Case of Big Datasets (or a Bit of Shamanism. Part 2)](#optimizing-time)
* [Experiment](#experiment)
    * [Scores](#scores)
    * [Bank Creation](#bank-creation)
* [Postprocessing](#postprocessing)

In [1]:
# General imports

import dill
import itertools
import json
import numpy as np
import os
import pandas as pd
import sys

from enum import Enum
from scipy.stats import gaussian_kde
from matplotlib import pyplot as plt
from tqdm import tqdm
from typing import (
    Dict,
    Iterable,
)

%matplotlib inline

In [2]:
# Making `topnum` module visible for Python

sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
# Optimal number of topics

from topicnet.cooking_machine import Dataset

from topnum.data.vowpal_wabbit_text_collection import VowpalWabbitTextCollection
from topnum.scores import (
    PerplexityScore,
    SparsityPhiScore,
    SparsityThetaScore,
)
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.scores._base_coherence_score import (
    SpecificityEstimationMethod,
    TextType,
    WordTopicRelatednessType,
)
from topnum.scores.intratext_coherence_score import ComputationMethod
from topnum.search_methods import TopicBankMethod
from topnum.search_methods.topic_bank.topic_bank import TopicBank
from topnum.search_methods.topic_bank.one_model_train_funcs import (
    default_train_func,

    # Functions below are not used (but could have been)

#     regularization_train_func,
#     specific_initial_phi_train_func,
#     background_topics_train_func,

)

## Data<a id="data"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Loading data from disk, creating batches, dictionary, gathering cooccurrence statistics...

In [4]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [5]:
sorted(os.listdir(DATA_FOLDER_PATH))

['20NG.csv',
 '20NG__internals',
 'Brown',
 'Brown_BOW.csv',
 'Brown_NOOW.csv',
 'MKB10.csv',
 'MKB10__internals',
 'RTL_Wiki.csv',
 'RTL_Wiki_person.csv',
 'RTL_Wiki_person__internals',
 'Reuters',
 'Reuters_BOW.csv',
 'Reuters_NOOW.csv',
 'WikiRef-220',
 '__init__.py',
 '__pycache__',
 'api.py',
 'postnauka.csv',
 'postnauka__internals',
 'ruwiki_good.txt',
 'ruwiki_good__internals',
 'wiki_ref220_bow.csv',
 'wiki_ref220_natural_order.csv']

In [7]:
class DatasetName(Enum):
    POSTNAUKA = 'Post_Science'
    # REUTERS = 'Reuters'
    # BROWN = 'Brown'
    TWENTY_NEWSGROUPS = '20_Newsgroups'
    GOOD_RU_WIKI = 'Good_RU_Wiki'
    RTL_WIKI_PERSON = 'RTL_Wiki_Person'

In [8]:
DATASET_NAME_TO_DATASET_FILE_PATH = {
    DatasetName.POSTNAUKA: os.path.join(
        DATA_FOLDER_PATH, 'postnauka.csv'
    ),
    # DatasetName.REUTERS: os.path.join(
    #     DATA_FOLDER_PATH, 'Reuters.csv'
    # ),
    # DatasetName.BROWN: os.path.join(
    #     DATA_FOLDER_PATH, 'Brown.csv'
    # ),
    DatasetName.TWENTY_NEWSGROUPS: os.path.join(
        DATA_FOLDER_PATH, '20NG.csv'
    ),
    # DatasetName.AG_NEWS: os.path.join(
    #     DATA_FOLDER_PATH, 'AG_News.csv'
    # ),
    # DatasetName.WATAN: os.path.join(
    #     DATA_FOLDER_PATH, 'Watan2004.csv'
    # ),
    # DatasetName.HABRAHABR: os.path.join(
    #     DATA_FOLDER_PATH, 'Habrahabr.csv'
    # ),
    DatasetName.GOOD_RU_WIKI: os.path.join(
        DATA_FOLDER_PATH, 'ruwiki_good.txt'
    ),
    DatasetName.RTL_WIKI_PERSON: os.path.join(
        DATA_FOLDER_PATH, 'RTL_Wiki_person.csv'
    ),
}

In [9]:
DATASET_NAME = DatasetName.RTL_WIKI_PERSON  # select a dataset here

DATASET_FILE_PATH = DATASET_NAME_TO_DATASET_FILE_PATH[DATASET_NAME]

Checking if all OK with data, what modalities does the collection have.

In [10]:
! head -n 2 $DATASET_FILE_PATH

,id,raw_text,vw_text
0,İsmet_İnönü,"Mustafa İsmet İnönü (September 24 1884 – December 25, 1973) was a Turkish Army General, TSK Genel Kurmay Baskanlari  Prime Minister and the second President of the Republic of Turkey. He is widely referred to as ""Milli Şef"" (National Chief), a title he bestowed upon himself  when he was elected as the President of Turkey in 1938.  Family and early life He was born in İzmir to a family originally from Malatya with mixed Turkish-Kurdish heritage. The Young Turks – Children of the Borderlands? - Erik Jan Zürcher (Universiteit Leiden)  Ismet Inonu: The Making of a Turkish Statesman - Metin Heper / Brill Academic Publishers  His father was Hacı Reşid Bey, a member of the Ottoman bureaucracy, an examining magistrate born in Malatya, and his mother was Cevriye Hanım, daughter of Russo-Turkish War refugees from Bulgaria. Due to his father's assignments, the family moved from one city to another. Thus, İsmet İnönü completed his primary education in Sivas.  

In [11]:
def get_dataset_internals_folder_path(dataset_name: DatasetName) -> str:
    return os.path.join('.', dataset_name.value + '__internals')

In [12]:
DATASET_INTERNALS_FOLDER_PATH = get_dataset_internals_folder_path(DATASET_NAME)

In [13]:
DATASET_INTERNALS_FOLDER_PATH

'./RTL_Wiki_Person__internals'

In [14]:
%%time

# If using really big datasets (like Habrahabr),
# one may need to set this equal `False`
KEEP_DATASET_IN_MEMORY = True

DATASET = Dataset(
    DATASET_FILE_PATH,
    internals_folder_path=DATASET_INTERNALS_FOLDER_PATH,
    keep_in_memory=KEEP_DATASET_IN_MEMORY,
)

CPU times: user 4.77 s, sys: 380 ms, total: 5.15 s
Wall time: 4.35 s


Looking what is inside dataset's folder

In [15]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['vw.txt', 'batches', 'dict.dict']

Creating batches

In [16]:
DATASET.get_batch_vectorizer()

artm.BatchVectorizer(data_path="./RTL_Wiki_Person__internals/batches", num_batches=2)

In [17]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['vw.txt', 'batches', 'dict.dict']

In [18]:
if KEEP_DATASET_IN_MEMORY:
    DOCUMENTS = list(DATASET._data.index)
else:
    DOCUMENTS = list(DATASET._data_index)

NUM_DOCUMENTS = len(DOCUMENTS)

print(f'Num documents: {NUM_DOCUMENTS}')

Num documents: 1201


Let's look at some text samples

In [19]:
DATASET._data.head()

,Unnamed: 0,id,raw_text,vw_text
id,,,,
İsmet_İnönü,0,İsmet_İnönü,Mustafa İsmet İnönü (September 24 1884 – Decem...,İsmet_İnönü |@lemmatized mustafa:2 smet:7 nönü...
Clara_Petacci,1,Clara_Petacci,Clara Petacci (Claretta Petacci) (28 February ...,Clara_Petacci |@lemmatized clara:5 petacci:15 ...
Jack_Ruby,2,Jack_Ruby,"Jacob Rubenstein (March 25, 1911 – January 3, ...",Jack_Ruby |@lemmatized jacob:2 rubenstein:5 ma...
Knud_Rasmussen,3,Knud_Rasmussen,"Knud Johan Victor Rasmussen (June 7, 1879–Dece...",Knud_Rasmussen |@lemmatized knud:15 johan:3 vi...
Gerald_Schroeder,4,Gerald_Schroeder,"Gerald L. Schroeder is a scientist, author, an...",Gerald_Schroeder |@lemmatized gerald:4 l:1 sch...


In [20]:
DATASET.get_possible_modalities()

{'@bigram', '@lemmatized'}

In [21]:
MAIN_MODALITY = '@lemmatized'

In [22]:
DATASET.get_dictionary()

artm.Dictionary(name=d57e2977-ac95-4e91-81f7-ea52db69cceb, num_entries=124241)

In [23]:
dictionary = DATASET.get_dictionary()

In [24]:
print(dictionary)

for modality in DATASET.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=d57e2977-ac95-4e91-81f7-ea52db69cceb, num_entries=124241)


In [25]:
dictionary.filter(min_df=2, max_df_rate=0.5)

artm.Dictionary(name=d57e2977-ac95-4e91-81f7-ea52db69cceb, num_entries=37739)

In [26]:
DATASET._cached_dict = dictionary

In [27]:
DATASET.get_dictionary()

artm.Dictionary(name=d57e2977-ac95-4e91-81f7-ea52db69cceb, num_entries=37739)

In [28]:
import scipy

from typing import List

from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)

In [29]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices=None,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        T, W = phi.shape
        # T = len(topic_indices)
        topic_indices = list(range(T))

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            # print(top, phi.shape, doc_co_occurrences.shape)
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [30]:
%%time

occurences, co_occurences = calc_doc_occurrences(DATASET, MAIN_MODALITY)

CPU times: user 7.64 s, sys: 312 ms, total: 7.95 s
Wall time: 7.89 s


In [31]:
co_occurences.shape

(37739, 37739)

In [32]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    DATASET.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [33]:
import copy


class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    @property
    def name(self):
        return self._name

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

    def compute(
            self,
            model,
            topics: List[str] = None,
            documents: List[str] = None) -> Dict[str, float]:

        values = self.call_by_topic(model)

        phi = model.get_phi()

        if topics is None:
            topics = list(phi.columns)

            if hasattr(model, 'has_bcg'):
                print(f'Detected bcg topics! Skipping for coherence computation (and will have {len(topics) - 1} topics).')

                topics = topics[:-1]
        else:
            assert False

        index2topic = {phi.columns.get_loc(t): t for t in topics}
        topic2index = {t: i for i, t in index2topic.items()}

        if hasattr(model, 'has_bcg'):
            assert list(index2topic.keys()) == list(values.keys())[:-1]
        else:
            assert list(index2topic.keys()) == list(values.keys())

        result = {
            t: float(values[topic2index[t]])
            for t in topics
        }

        assert len(result) == len(index2topic)

        return result

    def _attach(self, model: TopicModel):
        if self._name in model.custom_scores:
            print(
                f'Score with such name "{self._name}" already attached to model!'
                f' So rewriting it...'
                f' All model\'s custom scores: {list(model.custom_scores.keys())}'
            )

        # TODO: TopicModel should provide ability to add custom scores
        model.custom_scores[self.name] = copy.deepcopy(self)

## Experiment<a id="experiment"></a>

Finally we are getting to the main part!)

### Scores (for Topics and Models)<a id="scores"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we define a lot of scores (which mainly differ in initial parameters).

In [34]:
ONE_MODEL_NUM_TOPICS = 20
NUM_TOP_WORDS = 20

In [35]:
top = NUM_TOP_WORDS
target_topic_indices = list(range(ONE_MODEL_NUM_TOPICS))

coherence_score = TopTokenCoherence(
    name=f'coherence_{top}',
    func=create_pmi_top_function(
        occurences, co_occurences,
        DATASET.get_dataset().shape[0], [top],
        # topic_indices=target_topic_indices,
        co_occurrences_smooth=1e-2,
    )
)

diversity_scores = [
    DiversityScore(
        name=f'diversity_{metric}',
        metric=metric,
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

Other coherence score variations

And a pair of default ARTM scores (these ones are fast)

In [36]:
other_scores = [
    PerplexityScore(
        name='perplexity'
    ),
]

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [37]:
NUM_ITERATIONS = 20

In [38]:
seed = 0

In [39]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = default_train_func  # default train func

In [40]:
DATASET_INTERNALS_FOLDER_PATH

'./RTL_Wiki_Person__internals'

In [41]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [42]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls -alh $DATASET_INTERNALS_FOLDER_PATH

./RTL_Wiki_Person__internals
total 17M
drwxrwxr-x 3 alekseev_v mil_lab 4,0K мар 25 15:05 .
drwxrwxr-x 9 alekseev_v mil_lab 4,0K мар 25 15:06 ..
drwxrwxr-x 2 alekseev_v mil_lab 4,0K мар 25 15:05 batches
-rw-rw-r-- 1 alekseev_v mil_lab 4,5M мар 25 15:05 dict.dict
-rw-rw-r-- 1 alekseev_v mil_lab  12M мар 25 15:05 vw.txt


In [43]:
SEARCH_RESULTS_FOLDER_PATH

'./RTL_Wiki_Person__internals/result'

In [44]:
! ls $SEARCH_RESULTS_FOLDER_PATH

ls: cannot access './RTL_Wiki_Person__internals/result': No such file or directory


In [45]:
BANK_FOLDER_PATH

'./RTL_Wiki_Person__internals/result/bank__0'

In [46]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [47]:
seed

0

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [48]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = coherence_score,
    other_topic_scores = [],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 90,

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

In [49]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [50]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls $DATASET_INTERNALS_FOLDER_PATH

./RTL_Wiki_Person__internals
batches  dict.dict  result  vw.txt


In [51]:
optimizer._save_file_path

'./RTL_Wiki_Person__internals/result/search_result__0.json'

In [52]:
optimizer._topic_bank._path

'./RTL_Wiki_Person__internals/result/bank__0'

Fulfilling the search (get ready for a really long process!):

In [53]:
%%time

optimizer.search_for_optimum(DATASET)

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.39it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 30360.240234375, 'coherence_20': 0.9716440986349368, 'diversity_euclidean': 0.04342937816150277, 'diversity_jensenshannon': 0.6270041534345829, 'diversity_hellinger': 0.7253933383944854, 'diversity_cosine': 0.8223334691446338, 'perplexity': 30360.240234375, 'ppl_fair': 30360.240234375, 'ppl_cheatty': 6915.70751953125}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.97it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 23742.203125, 'coherence_20': 1.0197347309602016, 'diversity_euclidean': 0.04188082299930829, 'diversity_jensenshannon': 0.6134521511805038, 'diversity_hellinger': 0.7122789638414443, 'diversity_cosine': 0.7813183215957394, 'perplexity': 23742.203125, 'ppl_fair': 23742.203125, 'ppl_cheatty': 6654.400390625}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.86it/s]


...

Bank scores: {'perplexity_score': 16778.28125, 'coherence_20': 0.9837518057052368, 'diversity_euclidean': 0.0407833412820179, 'diversity_jensenshannon': 0.6126063133331306, 'diversity_hellinger': 0.7074486223274036, 'diversity_cosine': 0.7716645923686808, 'perplexity': 16778.28125, 'ppl_fair': 16778.28125, 'ppl_cheatty': 6275.625}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.73it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 16778.28125, 'coherence_20': 0.9837518057052368, 'diversity_euclidean': 0.0407833412820179, 'diversity_jensenshannon': 0.6126063133331306, 'diversity_hellinger': 0.7074486223274036, 'diversity_cosine': 0.7716645923686808, 'perplexity': 16778.28125, 'ppl_fair': 16778.28125, 'ppl_cheatty': 6275.625}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.79it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 16778.28125, 'coherence_20': 0.9837518057052368, 'diversity_euclidean': 0.0407833412820179, 'diversity_jensenshannon': 0.6126063133331306, 'diversity_hellinger': 0.7074486223274036, 'diversity_cosine': 0.7716645923686808, 'perplexity': 16778.28125, 'ppl_fair': 16778.28125, 'ppl_cheatty': 6275.625}
100%|███████████████████████████████████████████████| 20/20 [21:47<00:00, 65.37s/it]
CPU times: user 25min 13s, sys: 53.1 s, total: 26min 6s
Wall time: 21min 47s


What topics we have in bank

In [54]:
optimizer._topic_bank.view_topics().head()

topic_0   topic_1  topic_2  topic_3
@lemmatized hazardous      0.000000  0.000000      0.0      0.0
            expostulation  0.000000  0.000000      0.0      0.0
            hearse         0.000000  0.000016      0.0      0.0
            valkyrie       0.000012  0.000000      0.0      0.0
            sagebrush      0.000000  0.000000      0.0      0.0

In [55]:
bank_topics = optimizer._topic_bank.view_topics()

In [56]:
bank_topics.shape

(37739, 4)

In [57]:
bank_topics.head()

topic_0   topic_1  topic_2  topic_3
@lemmatized hazardous      0.000000  0.000000      0.0      0.0
            expostulation  0.000000  0.000000      0.0      0.0
            hearse         0.000000  0.000016      0.0      0.0
            valkyrie       0.000012  0.000000      0.0      0.0
            sagebrush      0.000000  0.000000      0.0      0.0

In [70]:
bank_topics['topic_0'].sort_values(ascending=False)[:20]

@lemmatized  roman          0.013310
             emperor        0.010589
             constantine    0.010339
             empire         0.006744
             rome           0.006158
             augustus       0.005679
             nero           0.005357
             p              0.004722
             reign          0.004370
             caesar         0.004049
             claudius       0.004032
             diocletian     0.003928
             caligula       0.003853
             julian         0.003545
             barnes         0.003446
             tacitus        0.003309
             göring         0.003194
             power          0.003078
             army           0.003078
             city           0.003027
Name: topic_0, dtype: float64

And topic scores

In [59]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3
kernel_size,4578.000000,5459.000000,4873.000000,4568.000000
coherence_20,1.027805,0.915483,1.115916,0.875803
distance_to_nearest,0.000000,0.825092,0.747386,0.801806


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [60]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [61]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [71]:
optimizer._result['num_bank_topics']

[2, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]

In [72]:
len(optimizer._result['bank_topic_scores'])

20

In [73]:
optimizer._result

{'optimum': 4,
 'optimum_std': 5.0,
 'bank_scores': [{'perplexity_score': 30360.240234375,
   'coherence_20': 0.9716440986349368,
   'diversity_euclidean': 0.04342937816150277,
   'diversity_jensenshannon': 0.6270041534345829,
   'diversity_hellinger': 0.7253933383944854,
   'diversity_cosine': 0.8223334691446338,
   'perplexity': 30360.240234375,
   'ppl_fair': 30360.240234375,
   'ppl_cheatty': 6915.70751953125},
  {'perplexity_score': 23742.203125,
   'coherence_20': 1.0197347309602016,
   'diversity_euclidean': 0.04188082299930829,
   'diversity_jensenshannon': 0.6134521511805038,
   'diversity_hellinger': 0.7122789638414443,
   'diversity_cosine': 0.7813183215957394,
   'perplexity': 23742.203125,
   'ppl_fair': 23742.203125,
   'ppl_cheatty': 6654.400390625},
  {'perplexity_score': 23742.203125,
   'coherence_20': 1.0197347309602016,
   'diversity_euclidean': 0.04188082299930829,
   'diversity_jensenshannon': 0.6134521511805038,
   'diversity_hellinger': 0.7122789638414443,
   'd

In [74]:
optimizer._result['bank_topic_scores'][0]

[{'kernel_size': 4578,
  'coherence_20': 1.027805102937893,
  'distance_to_nearest': 0.0},
 {'kernel_size': 5459,
  'coherence_20': 0.9154830943319806,
  'distance_to_nearest': 0.8250924654498791}]

In [75]:
optimizer._result['model_topic_scores']

[[{'kernel_size': 5435, 'coherence_20': 0.5603347657789858},
  {'kernel_size': 5110, 'coherence_20': 0.5431526298456942},
  {'kernel_size': 4578,
   'coherence_20': 1.027805102937893,
   'distance_to_nearest': 0.0},
  {'kernel_size': 5216, 'coherence_20': 0.6887570594579876},
  {'kernel_size': 4408, 'coherence_20': 0.5417173632646616},
  {'kernel_size': 4713, 'coherence_20': 0.482951624416885},
  {'kernel_size': 5057, 'coherence_20': 0.5494108141725347},
  {'kernel_size': 5008, 'coherence_20': 0.3908464817009172},
  {'kernel_size': 5292, 'coherence_20': 0.49182488666950436},
  {'kernel_size': 3869, 'coherence_20': 0.38324725763252804},
  {'kernel_size': 4927, 'coherence_20': 0.44415162080675186},
  {'kernel_size': 4760, 'coherence_20': 0.5594541276363032},
  {'kernel_size': 4955, 'coherence_20': 0.48628548352487966},
  {'kernel_size': 5302, 'coherence_20': 0.5834474807630564},
  {'kernel_size': 4918, 'coherence_20': 0.3817047960145765},
  {'kernel_size': 5459,
   'coherence_20': 0.9154

In [76]:
optimizer._result['model_scores'][0]

{'perplexity_score': 3568.923095703125,
 'coherence_20': 0.5462817220091145,
 'diversity_euclidean': 0.04151903238828226,
 'diversity_jensenshannon': 0.5967508853408056,
 'diversity_hellinger': 0.6839170513341035,
 'diversity_cosine': 0.7379828421055898,
 'perplexity': 3568.923095703125}

In [77]:
optimizer._result['model_scores'][1]

{'perplexity_score': 3508.67333984375,
 'coherence_20': 0.5581555265038253,
 'diversity_euclidean': 0.0417398762907053,
 'diversity_jensenshannon': 0.5966318698680039,
 'diversity_hellinger': 0.6836359012895517,
 'diversity_cosine': 0.7409157511779757,
 'perplexity': 3508.67333984375}

In [78]:
sum(s['coherence_20'] for s in optimizer._result['bank_topic_scores'][-1]) / 4

0.9837518057052368

In [79]:
len(optimizer._result['bank_scores'])

20

In [80]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 16778.28125,
 'coherence_20': 0.9837518057052368,
 'diversity_euclidean': 0.0407833412820179,
 'diversity_jensenshannon': 0.6126063133331306,
 'diversity_hellinger': 0.7074486223274036,
 'diversity_cosine': 0.7716645923686808,
 'perplexity': 16778.28125,
 'ppl_fair': 16778.28125,
 'ppl_cheatty': 6275.625}

In [81]:
optimizer._result['bank_topic_scores'][-1]

[{'kernel_size': 4578,
  'coherence_20': 1.027805102937893,
  'distance_to_nearest': 0.0},
 {'kernel_size': 5459,
  'coherence_20': 0.9154830943319806,
  'distance_to_nearest': 0.8250924654498791},
 {'kernel_size': 4873,
  'coherence_20': 1.1159159956107307,
  'distance_to_nearest': 0.7473857947211626},
 {'kernel_size': 4568,
  'coherence_20': 0.8758030299403429,
  'distance_to_nearest': 0.8018063294358485}]

In [82]:
import artm
from topnum.model_constructor import KnownModel, init_plsa
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    transform_regularizer,
)
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    init_model,
)

def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )
    model.has_bcg = True  # TODO: only if init_bcg_sparse_model

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [83]:
def artm_train_func(
        dataset: Dataset,
        model_number: int,
        num_topics: int,
        num_fit_iterations: int,
        scores: List = None,
        **kwargs) -> TopicModel:
    """

    Additional Parameters
    ---------------------
    kwargs
        Some params for `_get_topic_model`, such as `cache_theta` and `num_processors`
    """

    topic_model = init_model_from_family(
        family='ARTM',
        dataset=DATASET,
        main_modality=MAIN_MODALITY,
        num_topics=ONE_MODEL_NUM_TOPICS,
        seed=model_number,
        model_params={
            'decorrelation_tau': 0.01,  # best values
            'smooth_bcg_tau': 0.05,
            'sparse_sp_tau': -0.05,
        }
    )

    num_fit_iterations_with_scores = 1

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=max(0, num_fit_iterations - num_fit_iterations_with_scores)
    )
    _fit_model_with_scores(
        topic_model,
        DATASET,
        scores,
        num_fit_iterations=num_fit_iterations_with_scores
    )

    return topic_model


def _fit_model_with_scores(
        topic_model: TopicModel,
        dataset: Dataset,
        scores: List = None,
        num_fit_iterations: int = 1):

    if scores is not None:
        for score in scores:
            score._attach(topic_model)

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=num_fit_iterations
    )

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [84]:
NUM_ITERATIONS = 20

In [85]:
seed = 0

In [86]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = artm_train_func  # default train func

In [87]:
DATASET_INTERNALS_FOLDER_PATH

'./RTL_Wiki_Person__internals'

In [88]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result2'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [89]:
SEARCH_RESULTS_FOLDER_PATH

'./RTL_Wiki_Person__internals/result2'

In [90]:
BANK_FOLDER_PATH

'./RTL_Wiki_Person__internals/result2/bank__0'

In [91]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [92]:
seed

0

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [93]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = coherence_score,
    other_topic_scores = [],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 0.7010941774163096,  # DIFF ALSO HERE

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

/home/alekseev_v/projects/iterative/../OptimalNumberOfTopics/topnum/search_methods/topic_bank/topic_bank_method.py:208: UserWarning: topic_score_threshold_percentile 0.7010941774163096 is less than one! It is expected to be in [0, 100]. Are you sure you want to proceed (yes/no)?
  warnings.warn(


In [94]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [95]:
! ls ./Post_Science__internals

batches    _result  _result2  result_unfiltered_dict
dict.dict  result   result2   vw.txt


In [96]:
optimizer._save_file_path

'./RTL_Wiki_Person__internals/result2/search_result__0.json'

In [97]:
optimizer._topic_bank._path

'./RTL_Wiki_Person__internals/result2/bank__0'

Fulfilling the search (get ready for a really long process!):

In [98]:
%%time

optimizer.search_for_optimum(DATASET)

  0%|                                                        | 0/20 [00:00<?, ?it/s]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).

  0%|                                                         | 0/1 [00:00<?, ?it/s]Detected bcg topics! Skipping for coherence computation (and will have 20 topics).

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.70it/s]
Skipping saving scores for bcg topic


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 31523.9765625, 'coherence_20': 0.989731201194739, 'diversity_euclidean': 0.05561582019921126, 'diversity_jensenshannon': 0.6749729441821608, 'diversity_hellinger': 0.7901205376902046, 'diversity_cosine': 0.8629070383588524, 'perplexity': 31523.9765625, 'ppl_fair': 31523.9765625, 'ppl_cheatty': 6687.73291015625}
  5%|██▍                                             | 1/20 [01:14<23:36, 74.56s/it]No spec topics
No spec topics


...

Bank scores: {'perplexity_score': 13329.8876953125, 'coherence_20': 0.9058404227165606, 'diversity_euclidean': 0.05184720024244504, 'diversity_jensenshannon': 0.6500697527850625, 'diversity_hellinger': 0.7586476679507803, 'diversity_cosine': 0.8126390939253648, 'perplexity': 13329.8876953125, 'ppl_fair': 13329.8876953125, 'ppl_cheatty': 5741.56787109375}
 95%|████████████████████████████████████████████▋  | 19/20 [28:26<01:34, 94.07s/it]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).

  0%|                                                         | 0/1 [00:00<?, ?it/s]Detected bcg topics! Skipping for coherence computation (and will have 20 topics).

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.63it/s]
Skipping saving scores for bcg topic


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 13329.8876953125, 'coherence_20': 0.9058404227165606, 'diversity_euclidean': 0.05184720024244504, 'diversity_jensenshannon': 0.6500697527850625, 'diversity_hellinger': 0.7586476679507803, 'diversity_cosine': 0.8126390939253648, 'perplexity': 13329.8876953125, 'ppl_fair': 13329.8876953125, 'ppl_cheatty': 5741.56787109375}
100%|███████████████████████████████████████████████| 20/20 [29:58<00:00, 89.93s/it]
CPU times: user 35min, sys: 1min 5s, total: 36min 6s
Wall time: 29min 58s


In [113]:
optimizer._main_modality

'@lemmatized'

What topics we have in bank

In [114]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1  topic_2  topic_3  topic_4  \
@lemmatized hazardous          0.0      0.0      0.0      0.0      0.0   
            expostulation      0.0      0.0      0.0      0.0      0.0   
            hearse             0.0      0.0      0.0      0.0      0.0   
            valkyrie           0.0      0.0      0.0      0.0      0.0   
            sagebrush          0.0      0.0      0.0      0.0      0.0   

                           topic_5  topic_6  topic_7  topic_8  topic_9  
@lemmatized hazardous          0.0      0.0      0.0      0.0      0.0  
            expostulation      0.0      0.0      0.0      0.0      0.0  
            hearse             0.0      0.0      0.0      0.0      0.0  
            valkyrie           0.0      0.0      0.0      0.0      0.0  
            sagebrush          0.0      0.0      0.0      0.0      0.0

In [115]:
bank_topics = optimizer._topic_bank.view_topics()

In [116]:
bank_topics.shape

(37739, 10)

In [117]:
bank_topics.head()

topic_0  topic_1  topic_2  topic_3  topic_4  \
@lemmatized hazardous          0.0      0.0      0.0      0.0      0.0   
            expostulation      0.0      0.0      0.0      0.0      0.0   
            hearse             0.0      0.0      0.0      0.0      0.0   
            valkyrie           0.0      0.0      0.0      0.0      0.0   
            sagebrush          0.0      0.0      0.0      0.0      0.0   

                           topic_5  topic_6  topic_7  topic_8  topic_9  
@lemmatized hazardous          0.0      0.0      0.0      0.0      0.0  
            expostulation      0.0      0.0      0.0      0.0      0.0  
            hearse             0.0      0.0      0.0      0.0      0.0  
            valkyrie           0.0      0.0      0.0      0.0      0.0  
            sagebrush          0.0      0.0      0.0      0.0      0.0

In [119]:
bank_topics['topic_7'].sort_values(ascending=False)[:20]

@lemmatized  philosophy       0.014656
             language         0.006478
             logic            0.005423
             russell          0.005302
             idea             0.005024
             hegel            0.005021
             peirce           0.005018
             science          0.004771
             philosopher      0.004757
             theory           0.004736
             human            0.004628
             austen           0.004608
             philosophical    0.003994
             aristotle        0.003845
             object           0.003833
             thing            0.003604
             p                0.003517
             thought          0.003460
             mind             0.003279
             cambridge        0.003254
Name: topic_7, dtype: float64

And topic scores

In [120]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9
kernel_size,3987.000000,3735.000000,4067.000000,4045.000000,3569.000000,3594.000000,4160.000000,3367.000000,3560.000000,3799.000000
coherence_20,1.284042,0.722904,0.732766,0.972763,0.782008,1.063218,0.748290,0.909087,1.002204,0.841123
distance_to_nearest,0.000000,0.860819,0.837529,0.766088,0.679722,0.512943,0.745597,0.580841,0.678975,0.537332


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [121]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [122]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [123]:
optimizer._result['num_bank_topics']

[4, 5, 5, 5, 6, 6, 6, 6, 7, 8, 8, 8, 8, 9, 10, 10, 10, 10, 10, 10]

In [124]:
len(optimizer._result['bank_topic_scores'])

20

In [125]:
optimizer._result

{'optimum': 10,
 'optimum_std': 5.0,
 'bank_scores': [{'perplexity_score': 31523.9765625,
   'coherence_20': 0.989731201194739,
   'diversity_euclidean': 0.05561582019921126,
   'diversity_jensenshannon': 0.6749729441821608,
   'diversity_hellinger': 0.7901205376902046,
   'diversity_cosine': 0.8629070383588524,
   'perplexity': 31523.9765625,
   'ppl_fair': 31523.9765625,
   'ppl_cheatty': 6687.73291015625},
  {'perplexity_score': 22795.0234375,
   'coherence_20': 0.9383382449021079,
   'diversity_euclidean': 0.05283096530416507,
   'diversity_jensenshannon': 0.6665119479987796,
   'diversity_hellinger': 0.7791928389577456,
   'diversity_cosine': 0.8481243646532777,
   'perplexity': 22795.0234375,
   'ppl_fair': 22795.0234375,
   'ppl_cheatty': 6460.900390625},
  {'perplexity_score': 22795.0234375,
   'coherence_20': 0.9383382449021079,
   'diversity_euclidean': 0.05283096530416507,
   'diversity_jensenshannon': 0.6665119479987796,
   'diversity_hellinger': 0.7791928389577456,
   'div

In [111]:
optimizer._result['bank_topic_scores']

[[{'kernel_size': 3987,
   'coherence_20': 1.284041737255726,
   'distance_to_nearest': 0.0},
  {'kernel_size': 3715,
   'coherence_20': 0.9103697786572175,
   'distance_to_nearest': 0.8389113266096888},
  {'kernel_size': 3735,
   'coherence_20': 0.7229039920008129,
   'distance_to_nearest': 0.8608185493321843},
  {'kernel_size': 3970,
   'coherence_20': 1.0416092968651998,
   'distance_to_nearest': 0.8743664527246292}],
 [{'kernel_size': 3987,
   'coherence_20': 1.284041737255726,
   'distance_to_nearest': 0.0},
  {'kernel_size': 3715,
   'coherence_20': 0.9103697786572175,
   'distance_to_nearest': 0.8389113266096888},
  {'kernel_size': 3735,
   'coherence_20': 0.7229039920008129,
   'distance_to_nearest': 0.8608185493321843},
  {'kernel_size': 3970,
   'coherence_20': 1.0416092968651998,
   'distance_to_nearest': 0.8743664527246292},
  {'kernel_size': 4067,
   'coherence_20': 0.7327664197315834,
   'distance_to_nearest': 0.8375291531011442}],
 [{'kernel_size': 3987,
   'coherence_20

In [112]:
optimizer._result['model_scores'][0]

{'perplexity_score': 4044.370849609375,
 'coherence_20': 0.5698039218087506,
 'diversity_euclidean': 0.05401568145913427,
 'diversity_jensenshannon': 0.6563731926360971,
 'diversity_hellinger': 0.7635746116244427,
 'diversity_cosine': 0.8266555444400919,
 'perplexity': 4044.370849609375}

In [126]:
sum(s['coherence_20'] for s in optimizer._result['bank_topic_scores'][-1]) / 10

0.9058404227165608

In [127]:
len(optimizer._result['bank_scores'])

20

In [128]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 13329.8876953125,
 'coherence_20': 0.9058404227165606,
 'diversity_euclidean': 0.05184720024244504,
 'diversity_jensenshannon': 0.6500697527850625,
 'diversity_hellinger': 0.7586476679507803,
 'diversity_cosine': 0.8126390939253648,
 'perplexity': 13329.8876953125,
 'ppl_fair': 13329.8876953125,
 'ppl_cheatty': 5741.56787109375}

In [129]:
optimizer._result['model_topic_scores']

[[{'kernel_size': 3810, 'coherence_20': 0.5225908625149936},
  {'kernel_size': 3895, 'coherence_20': 0.5331450869041109},
  {'kernel_size': 3987,
   'coherence_20': 1.284041737255726,
   'distance_to_nearest': 0.0},
  {'kernel_size': 3896, 'coherence_20': 0.5261857613384362},
  {'kernel_size': 3860, 'coherence_20': 0.3796521580255667},
  {'kernel_size': 3654, 'coherence_20': 0.5656146718404065},
  {'kernel_size': 3844, 'coherence_20': 0.44309547614289096},
  {'kernel_size': 3919, 'coherence_20': 0.5484668820539618},
  {'kernel_size': 4084, 'coherence_20': 0.5739931717770947},
  {'kernel_size': 2794, 'coherence_20': 0.5111064072825383},
  {'kernel_size': 4275, 'coherence_20': 0.5617371534777974},
  {'kernel_size': 3470, 'coherence_20': 0.6047710619718488},
  {'kernel_size': 3392, 'coherence_20': 0.6510584572257663},
  {'kernel_size': 3715,
   'coherence_20': 0.9103697786572175,
   'distance_to_nearest': 0.8389113266096888},
  {'kernel_size': 3735,
   'coherence_20': 0.7229039920008129,


In [130]:
! echo $SEARCH_RESULTS_FOLDER_PATH
! ls $SEARCH_RESULTS_FOLDER_PATH

./RTL_Wiki_Person__internals/result2
bank__0  search_result__0.json


In [131]:
! ls Good_RU_Wiki__internals

batches  dict.dict  result  result2  vw.txt
